# Catastrophic Forgetting & Continual Learning

## Overview
Catastrophic forgetting occurs when a neural network forgets previously learned knowledge upon learning new information. This notebook covers detection, measurement, and mitigation strategies for continual learning scenarios.

**Key Topics:**
- Catastrophic forgetting problem
- Elastic Weight Consolidation (EWC)
- Experience replay buffers
- Data mixing strategies (80/20 rule)
- Continual learning methods
- Measuring forgetting
- Prevention strategies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict, deque
import copy

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Historical Context & Theory

### Evolution of Catastrophic Forgetting Research

**1989 - Discovery:**
- McCloskey & Cohen: "Catastrophic interference in connectionist networks"
- Showed neural networks completely forget old tasks when learning new ones
- Contrasted with gradual forgetting in human learning

**1990s - Early Solutions:**
- Rehearsal methods: store and replay old examples
- Pseudo-rehearsal: generate synthetic old data
- Limited by memory and computational constraints

**2000s - Theoretical Foundations:**
- French (1999): Stability-plasticity dilemma
- Need balance between learning new information and retaining old
- Biological inspiration from hippocampal replay

**2017 - EWC Breakthrough:**
- Kirkpatrick et al. (DeepMind): Elastic Weight Consolidation
- Protect important weights using Fisher Information
- Enabled sequential learning without catastrophic forgetting
- Demonstrated on Atari games and MNIST variants

**2017-2019 - Regularization Methods:**
- Learning without Forgetting (LwF): distillation-based
- Synaptic Intelligence: online importance estimation
- Memory Aware Synapses (MAS): output-based importance

**2019-2020 - Replay Methods:**
- Gradient Episodic Memory (GEM): constrained optimization
- Experience Replay: store representative examples
- DER++: Dark Experience Replay with logits

**2020-Present - LLM Era:**
- Continual pre-training of language models
- Domain adaptation without forgetting
- Data mixing strategies (80/20, 90/10 rules)
- LoRA and adapter-based methods reduce forgetting
- Instruction tuning: balancing new capabilities with base model knowledge

### Theoretical Foundations

**The Catastrophic Forgetting Problem:**

When learning task $B$ after task $A$:
$$\theta^* = \arg\min_{\theta} L_B(\theta)$$

This typically causes:
$$L_A(\theta^*) >> L_A(\theta_A)$$

Where $\theta_A$ was optimal for task $A$.

**Stability-Plasticity Dilemma:**
- **Plasticity:** Ability to learn new information
- **Stability:** Ability to retain old knowledge
- Trade-off: high plasticity → forgetting, high stability → can't learn new tasks

**Forgetting Metrics:**

1. **Backward Transfer (BWT):**
$$BWT = \frac{1}{T-1} \sum_{i=1}^{T-1} (R_{T,i} - R_{i,i})$$

Where $R_{j,i}$ is performance on task $i$ after learning task $j$.

2. **Forgetting Measure:**
$$F_i = \max_{j \in [1, T-1]} R_{j,i} - R_{T,i}$$

Maximum performance achieved minus final performance.

## 2. Mathematical Foundations

### Elastic Weight Consolidation (EWC)

**Core Idea:** Constrain important parameters from changing when learning new tasks.

**EWC Loss:**
$$L_{EWC}(\theta) = L_B(\theta) + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta^*_{A,i})^2$$

Where:
- $L_B(\theta)$ = loss on new task $B$
- $\lambda$ = regularization strength
- $F_i$ = Fisher Information for parameter $i$
- $\theta^*_A$ = optimal parameters after learning task $A$

**Fisher Information Matrix:**
$$F_i = E_{x \sim p_A} \left[ \left( \frac{\partial \log p(x|\theta)}{\partial \theta_i} \right)^2 \right]$$

Approximated using:
$$F_i \approx \frac{1}{N} \sum_{n=1}^N \left( \frac{\partial \log p(x_n|\theta^*_A)}{\partial \theta_i} \right)^2$$

### Gradient Episodic Memory (GEM)

**Constrained Optimization:**
Ensure gradients on new task don't increase loss on old tasks:

$$\min_{\theta} L_B(\theta)$$
$$\text{s.t. } \langle g_B, g_A \rangle \geq 0$$

Where $g_A, g_B$ are gradients on old and new tasks.

If constraint violated, project gradient:
$$g'_B = g_B - \frac{\langle g_B, g_A \rangle}{||g_A||^2} g_A$$

### Data Mixing

**80/20 Rule (Common Practice):**
$$L_{total} = 0.8 \cdot L_{new} + 0.2 \cdot L_{old}$$

**Dynamic Mixing with Temperature:**
$$P(\text{sample old}) = \frac{|D_{old}|^\tau}{|D_{old}|^\tau + |D_{new}|^\tau}$$

Typical: $\tau = 0.7$ to balance datasets.

## 3. Implementation: Elastic Weight Consolidation

In [ ]:
class EWC:
    """Elastic Weight Consolidation for continual learning."""
    
    def __init__(self, model: nn.Module, lambda_ewc: float = 0.4):
        """
        Args:
            model: Neural network model
            lambda_ewc: EWC regularization strength (typical: 0.1-10)
        """
        self.model = model
        self.lambda_ewc = lambda_ewc
        
        # Store parameters and Fisher information for each task
        self.task_params = {}  # task_id -> {name: param}
        self.task_fisher = {}  # task_id -> {name: fisher}
        
    def compute_fisher_information(
        self,
        task_id: str,
        dataloader: any,
        num_samples: int = 1000
    ):
        """
        Compute Fisher Information Matrix for current task.
        
        Args:
            task_id: Identifier for the task
            dataloader: Data loader for the task
            num_samples: Number of samples to use for estimation
        """
        fisher_dict = {}
        param_dict = {}
        
        # Initialize Fisher information
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                fisher_dict[name] = torch.zeros_like(param.data)
                param_dict[name] = param.data.clone()
        
        # Compute Fisher as expectation of squared gradients
        self.model.eval()
        samples_seen = 0
        
        for inputs, targets in dataloader:
            if samples_seen >= num_samples:
                break
            
            self.model.zero_grad()
            outputs = self.model(inputs)
            
            # For classification: sample from predicted distribution
            if outputs.dim() > 1:  # Classification
                log_probs = F.log_softmax(outputs, dim=-1)
                probs = F.softmax(outputs, dim=-1)
                
                # Sample labels from model's prediction
                sampled_labels = torch.multinomial(probs, 1).squeeze()
                
                # Compute log likelihood
                loss = F.nll_loss(log_probs, sampled_labels)
            else:  # Regression
                loss = F.mse_loss(outputs, targets)
            
            loss.backward()
            
            # Accumulate squared gradients
            for name, param in self.model.named_parameters():
                if param.requires_grad and param.grad is not None:
                    fisher_dict[name] += param.grad.data ** 2
            
            samples_seen += inputs.size(0)
        
        # Average over samples
        for name in fisher_dict:
            fisher_dict[name] /= samples_seen
        
        # Store for this task
        self.task_fisher[task_id] = fisher_dict
        self.task_params[task_id] = param_dict
        
        print(f"Computed Fisher information for task '{task_id}'")
        print(f"  Samples used: {samples_seen}")
        print(f"  Parameters tracked: {len(fisher_dict)}")
    
    def compute_ewc_loss(self) -> torch.Tensor:
        """
        Compute EWC regularization loss.
        
        Returns:
            EWC penalty summed over all previous tasks
        """
        ewc_loss = torch.tensor(0.0, device=next(self.model.parameters()).device)
        
        for task_id in self.task_fisher:
            fisher = self.task_fisher[task_id]
            old_params = self.task_params[task_id]
            
            for name, param in self.model.named_parameters():
                if param.requires_grad and name in fisher:
                    # EWC penalty: F_i * (theta_i - theta*_i)^2
                    fisher_weight = fisher[name]
                    old_param = old_params[name]
                    ewc_loss += (fisher_weight * (param - old_param) ** 2).sum()
        
        return (self.lambda_ewc / 2) * ewc_loss
    
    def get_importance_statistics(self) -> Dict[str, Dict[str, float]]:
        """Get statistics about parameter importance across tasks."""
        stats = {}
        
        for task_id, fisher in self.task_fisher.items():
            task_stats = {}
            all_values = torch.cat([f.flatten() for f in fisher.values()])
            
            task_stats['mean'] = all_values.mean().item()
            task_stats['std'] = all_values.std().item()
            task_stats['max'] = all_values.max().item()
            task_stats['median'] = all_values.median().item()
            task_stats['num_params'] = len(all_values)
            
            stats[task_id] = task_stats
        
        return stats

# Visualize Fisher Information
def visualize_fisher_importance(ewc: EWC):
    """Visualize parameter importance across tasks."""
    if not ewc.task_fisher:
        print("No Fisher information computed yet.")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Distribution of Fisher values per task
    ax1 = axes[0]
    for task_id, fisher in ewc.task_fisher.items():
        all_values = torch.cat([f.flatten() for f in fisher.values()]).cpu().numpy()
        # Log scale for better visualization
        log_values = np.log10(all_values + 1e-10)
        ax1.hist(log_values, bins=50, alpha=0.6, label=f'Task {task_id}')
    
    ax1.set_xlabel('log10(Fisher Information)')
    ax1.set_ylabel('Count')
    ax1.set_title('Distribution of Parameter Importance')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Comparison of importance statistics
    ax2 = axes[1]
    stats = ewc.get_importance_statistics()
    
    tasks = list(stats.keys())
    x = np.arange(len(tasks))
    width = 0.2
    
    means = [stats[t]['mean'] for t in tasks]
    maxs = [stats[t]['max'] for t in tasks]
    medians = [stats[t]['median'] for t in tasks]
    
    ax2.bar(x - width, means, width, label='Mean', alpha=0.8)
    ax2.bar(x, medians, width, label='Median', alpha=0.8)
    ax2.bar(x + width, maxs, width, label='Max', alpha=0.8)
    
    ax2.set_xlabel('Task')
    ax2.set_ylabel('Fisher Value')
    ax2.set_title('Parameter Importance Statistics')
    ax2.set_xticks(x)
    ax2.set_xticklabels(tasks)
    ax2.legend()
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('fisher_information_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

print("EWC implementation complete")

## 4. Experience Replay Buffer

In [ ]:
class ExperienceReplayBuffer:
    """Store representative examples from previous tasks."""
    
    def __init__(
        self,
        max_size: int = 10000,
        per_task_size: Optional[int] = None,
        sampling_strategy: str = 'reservoir'  # 'reservoir', 'balanced', 'recent'
    ):
        """
        Args:
            max_size: Maximum total examples to store
            per_task_size: Maximum examples per task (if None, use max_size)
            sampling_strategy: How to select examples to store
        """
        self.max_size = max_size
        self.per_task_size = per_task_size or max_size
        self.sampling_strategy = sampling_strategy
        
        # Storage: task_id -> list of (input, target) tuples
        self.task_buffers = defaultdict(list)
        self.task_counts = defaultdict(int)  # Total examples seen per task
        
    def add_example(self, task_id: str, input_data: any, target: any):
        """Add a single example to the buffer."""
        self.task_counts[task_id] += 1
        buffer = self.task_buffers[task_id]
        
        if self.sampling_strategy == 'reservoir':
            # Reservoir sampling: uniform probability over stream
            if len(buffer) < self.per_task_size:
                buffer.append((input_data, target))
            else:
                # Random replacement
                idx = np.random.randint(0, self.task_counts[task_id])
                if idx < self.per_task_size:
                    buffer[idx] = (input_data, target)
        
        elif self.sampling_strategy == 'recent':
            # Keep most recent examples
            if len(buffer) < self.per_task_size:
                buffer.append((input_data, target))
            else:
                buffer.pop(0)  # Remove oldest
                buffer.append((input_data, target))
        
        else:  # balanced
            buffer.append((input_data, target))
            # Trim if needed
            if len(buffer) > self.per_task_size:
                # Random removal to maintain balance
                idx = np.random.randint(0, len(buffer))
                buffer.pop(idx)
    
    def add_batch(self, task_id: str, inputs: torch.Tensor, targets: torch.Tensor):
        """Add a batch of examples."""
        for i in range(inputs.size(0)):
            self.add_example(task_id, inputs[i], targets[i])
    
    def sample_batch(
        self,
        batch_size: int,
        task_id: Optional[str] = None
    ) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
        """
        Sample a batch of replay examples.
        
        Args:
            batch_size: Number of examples to sample
            task_id: If specified, sample only from this task
        
        Returns:
            inputs, targets, task_ids
        """
        if task_id:
            tasks = [task_id]
        else:
            tasks = list(self.task_buffers.keys())
        
        if not tasks:
            return None, None, []
        
        # Sample tasks proportionally to buffer size
        task_sizes = {t: len(self.task_buffers[t]) for t in tasks}
        total_size = sum(task_sizes.values())
        
        if total_size == 0:
            return None, None, []
        
        task_probs = [task_sizes[t] / total_size for t in tasks]
        
        # Sample examples
        sampled_inputs = []
        sampled_targets = []
        sampled_tasks = []
        
        for _ in range(batch_size):
            # Sample task
            task = np.random.choice(tasks, p=task_probs)
            
            # Sample example from task
            buffer = self.task_buffers[task]
            if buffer:
                idx = np.random.randint(0, len(buffer))
                input_data, target = buffer[idx]
                sampled_inputs.append(input_data)
                sampled_targets.append(target)
                sampled_tasks.append(task)
        
        if not sampled_inputs:
            return None, None, []
        
        # Stack into tensors
        inputs = torch.stack(sampled_inputs)
        targets = torch.stack(sampled_targets)
        
        return inputs, targets, sampled_tasks
    
    def get_statistics(self) -> Dict[str, any]:
        """Get buffer statistics."""
        stats = {
            'total_stored': sum(len(b) for b in self.task_buffers.values()),
            'total_seen': sum(self.task_counts.values()),
            'num_tasks': len(self.task_buffers),
            'per_task': {}
        }
        
        for task_id in self.task_buffers:
            stats['per_task'][task_id] = {
                'stored': len(self.task_buffers[task_id]),
                'seen': self.task_counts[task_id],
                'retention_rate': len(self.task_buffers[task_id]) / max(1, self.task_counts[task_id])
            }
        
        return stats
    
    def visualize_buffer(self):
        """Visualize buffer contents and statistics."""
        stats = self.get_statistics()
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 1. Examples stored vs seen
        ax1 = axes[0]
        tasks = list(stats['per_task'].keys())
        stored = [stats['per_task'][t]['stored'] for t in tasks]
        seen = [stats['per_task'][t]['seen'] for t in tasks]
        
        x = np.arange(len(tasks))
        width = 0.35
        
        ax1.bar(x - width/2, stored, width, label='Stored', alpha=0.8)
        ax1.bar(x + width/2, seen, width, label='Seen', alpha=0.8)
        
        ax1.set_xlabel('Task')
        ax1.set_ylabel('Number of Examples')
        ax1.set_title('Replay Buffer: Stored vs Seen')
        ax1.set_xticks(x)
        ax1.set_xticklabels(tasks, rotation=45, ha='right')
        ax1.legend()
        ax1.grid(True, alpha=0.3, axis='y')
        
        # 2. Retention rates
        ax2 = axes[1]
        retention = [stats['per_task'][t]['retention_rate'] * 100 for t in tasks]
        
        ax2.bar(tasks, retention, alpha=0.8, color='coral')
        ax2.set_xlabel('Task')
        ax2.set_ylabel('Retention Rate (%)')
        ax2.set_title('Percentage of Examples Retained in Buffer')
        ax2.set_xticklabels(tasks, rotation=45, ha='right')
        ax2.grid(True, alpha=0.3, axis='y')
        ax2.set_ylim([0, 100])
        
        plt.tight_layout()
        plt.savefig('replay_buffer_statistics.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print statistics
        print("\nReplay Buffer Statistics:")
        print(f"Total examples stored: {stats['total_stored']:,}")
        print(f"Total examples seen: {stats['total_seen']:,}")
        print(f"Number of tasks: {stats['num_tasks']}")
        print(f"Overall retention rate: {stats['total_stored'] / max(1, stats['total_seen']) * 100:.2f}%")

print("Experience replay buffer implemented")

## 5. Data Mixing Strategies

In [ ]:
class ContinualLearningDataMixer:
    """Mix new and old data to prevent catastrophic forgetting."""
    
    def __init__(
        self,
        replay_buffer: ExperienceReplayBuffer,
        mixing_ratio: float = 0.2,  # Fraction of batch from replay
        strategy: str = 'fixed'  # 'fixed', 'adaptive', 'curriculum'
    ):
        """
        Args:
            replay_buffer: Experience replay buffer
            mixing_ratio: Fraction of batch from old tasks (0.2 = 80/20 rule)
            strategy: Mixing strategy
        """
        self.replay_buffer = replay_buffer
        self.mixing_ratio = mixing_ratio
        self.strategy = strategy
        
        # For adaptive strategy
        self.task_performance = defaultdict(list)
        self.current_step = 0
    
    def get_mixed_batch(
        self,
        new_inputs: torch.Tensor,
        new_targets: torch.Tensor,
        current_task: str
    ) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
        """
        Create mixed batch of new and replay examples.
        
        Args:
            new_inputs: Batch of new task inputs
            new_targets: Batch of new task targets
            current_task: ID of current task
        
        Returns:
            Mixed batch: (inputs, targets, task_ids)
        """
        batch_size = new_inputs.size(0)
        
        # Determine mixing ratio
        if self.strategy == 'adaptive':
            ratio = self._compute_adaptive_ratio(current_task)
        elif self.strategy == 'curriculum':
            ratio = self._compute_curriculum_ratio()
        else:
            ratio = self.mixing_ratio
        
        # Calculate split
        num_replay = int(batch_size * ratio)
        num_new = batch_size - num_replay
        
        # Sample new examples
        if num_new < batch_size:
            indices = torch.randperm(batch_size)[:num_new]
            new_inputs = new_inputs[indices]
            new_targets = new_targets[indices]
        
        # Sample replay examples
        if num_replay > 0:
            replay_inputs, replay_targets, replay_tasks = self.replay_buffer.sample_batch(num_replay)
            
            if replay_inputs is not None:
                # Combine batches
                inputs = torch.cat([new_inputs, replay_inputs], dim=0)
                targets = torch.cat([new_targets, replay_targets], dim=0)
                task_ids = [current_task] * num_new + replay_tasks
                
                # Shuffle
                perm = torch.randperm(len(task_ids))
                inputs = inputs[perm]
                targets = targets[perm]
                task_ids = [task_ids[i] for i in perm]
                
                return inputs, targets, task_ids
        
        # No replay samples available
        return new_inputs, new_targets, [current_task] * num_new
    
    def _compute_adaptive_ratio(self, current_task: str) -> float:
        """
        Adapt mixing ratio based on forgetting.
        Increase replay if old task performance drops.
        """
        # Check if we have performance history
        if not self.task_performance:
            return self.mixing_ratio
        
        # Compute average forgetting
        total_forgetting = 0
        num_old_tasks = 0
        
        for task_id, perf_history in self.task_performance.items():
            if task_id != current_task and len(perf_history) > 1:
                # Forgetting = max_perf - current_perf
                forgetting = max(perf_history) - perf_history[-1]
                total_forgetting += max(0, forgetting)  # Only positive forgetting
                num_old_tasks += 1
        
        if num_old_tasks == 0:
            return self.mixing_ratio
        
        avg_forgetting = total_forgetting / num_old_tasks
        
        # Increase replay ratio if forgetting is high
        # Forgetting of 0.1 (10%) -> increase ratio by 0.1
        adjusted_ratio = min(0.5, self.mixing_ratio + avg_forgetting)
        
        return adjusted_ratio
    
    def _compute_curriculum_ratio(self) -> float:
        """
        Gradually increase replay ratio as training progresses.
        Start with more new data, gradually mix in more replay.
        """
        # Linear increase from 0 to mixing_ratio over 10000 steps
        max_steps = 10000
        progress = min(1.0, self.current_step / max_steps)
        return self.mixing_ratio * progress
    
    def update_performance(self, task_id: str, performance: float):
        """Update task performance for adaptive mixing."""
        self.task_performance[task_id].append(performance)
        self.current_step += 1

# Visualize mixing strategies
def visualize_mixing_strategies():
    """Compare different mixing strategies over time."""
    steps = np.arange(0, 15000, 100)
    
    # Simulate different strategies
    fixed_ratios = [0.2] * len(steps)
    curriculum_ratios = [min(0.2, 0.2 * (s / 10000)) for s in steps]
    
    # Adaptive: simulate forgetting at step 5000
    adaptive_ratios = []
    base_ratio = 0.2
    for s in steps:
        if s < 5000:
            adaptive_ratios.append(base_ratio)
        else:
            # Simulate forgetting detection
            forgetting = 0.15 * (1 - np.exp(-(s - 5000) / 3000))
            adaptive_ratios.append(min(0.5, base_ratio + forgetting))
    
    plt.figure(figsize=(10, 6))
    plt.plot(steps, fixed_ratios, label='Fixed (80/20)', linewidth=2)
    plt.plot(steps, curriculum_ratios, label='Curriculum', linewidth=2)
    plt.plot(steps, adaptive_ratios, label='Adaptive', linewidth=2)
    
    plt.axhline(y=0.2, color='gray', linestyle='--', alpha=0.5, label='80/20 baseline')
    plt.axvline(x=5000, color='red', linestyle='--', alpha=0.5, label='Forgetting detected')
    
    plt.xlabel('Training Step')
    plt.ylabel('Replay Ratio (fraction of batch)')
    plt.title('Data Mixing Strategies for Continual Learning')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 0.6])
    
    plt.tight_layout()
    plt.savefig('mixing_strategies_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

visualize_mixing_strategies()
print("\nData mixing implementation complete")

## 6. Forgetting Measurement & Detection

In [ ]:
class ForgettingTracker:
    """Track and measure catastrophic forgetting across tasks."""
    
    def __init__(self, task_names: List[str]):
        self.task_names = task_names
        self.num_tasks = len(task_names)
        
        # Performance matrix: R[j,i] = performance on task i after learning task j
        self.performance_matrix = np.zeros((self.num_tasks, self.num_tasks))
        
        # Track when each task was learned
        self.task_order = []
        
        # Detailed history
        self.performance_history = defaultdict(list)
    
    def record_performance(
        self,
        current_task_idx: int,
        task_performances: Dict[str, float]
    ):
        """
        Record performance on all tasks after learning current task.
        
        Args:
            current_task_idx: Index of task just learned
            task_performances: Dict of {task_name: performance}
        """
        for i, task_name in enumerate(self.task_names):
            if task_name in task_performances:
                perf = task_performances[task_name]
                self.performance_matrix[current_task_idx, i] = perf
                self.performance_history[task_name].append(perf)
        
        if current_task_idx not in self.task_order:
            self.task_order.append(current_task_idx)
    
    def compute_backward_transfer(self) -> float:
        """
        Compute Backward Transfer (BWT).
        Measures how much learning new tasks affects old task performance.
        
        Negative BWT indicates forgetting.
        """
        if len(self.task_order) < 2:
            return 0.0
        
        bwt_sum = 0.0
        count = 0
        
        final_task_idx = self.task_order[-1]
        
        for i in range(len(self.task_order) - 1):
            task_idx = self.task_order[i]
            # Performance when task was learned vs. at the end
            initial_perf = self.performance_matrix[task_idx, task_idx]
            final_perf = self.performance_matrix[final_task_idx, task_idx]
            
            if initial_perf > 0:  # Avoid division by zero
                bwt_sum += (final_perf - initial_perf)
                count += 1
        
        return bwt_sum / max(1, count)
    
    def compute_forgetting_measure(self) -> Dict[str, float]:
        """
        Compute forgetting for each task.
        
        Forgetting = max_performance - final_performance
        """
        forgetting = {}
        
        for i, task_name in enumerate(self.task_names):
            if i in self.task_order[:-1]:  # Exclude current task
                # Get all performances for this task
                perfs = self.performance_matrix[self.task_order, i]
                perfs = perfs[perfs > 0]  # Filter out zeros
                
                if len(perfs) > 1:
                    max_perf = perfs.max()
                    final_perf = perfs[-1]
                    forgetting[task_name] = max_perf - final_perf
        
        return forgetting
    
    def compute_forward_transfer(self) -> float:
        """
        Compute Forward Transfer (FWT).
        Measures how much previous tasks help learning new tasks.
        
        Positive FWT indicates positive transfer.
        """
        # Would need single-task baseline performance
        # For now, return 0 (placeholder)
        return 0.0
    
    def visualize_forgetting(self):
        """Visualize catastrophic forgetting patterns."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1. Performance matrix heatmap
        ax1 = axes[0, 0]
        im = ax1.imshow(self.performance_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
        ax1.set_xlabel('Task Evaluated')
        ax1.set_ylabel('After Learning Task')
        ax1.set_title('Performance Matrix')
        ax1.set_xticks(range(self.num_tasks))
        ax1.set_yticks(range(self.num_tasks))
        ax1.set_xticklabels(self.task_names, rotation=45, ha='right')
        ax1.set_yticklabels(self.task_names)
        plt.colorbar(im, ax=ax1, label='Performance')
        
        # 2. Performance over time for each task
        ax2 = axes[0, 1]
        for task_name in self.task_names:
            history = self.performance_history[task_name]
            if history:
                ax2.plot(range(len(history)), history, marker='o', label=task_name, linewidth=2)
        ax2.set_xlabel('Learning Step (after task N)')
        ax2.set_ylabel('Performance')
        ax2.set_title('Task Performance Over Time')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Forgetting measure per task
        ax3 = axes[1, 0]
        forgetting = self.compute_forgetting_measure()
        if forgetting:
            tasks = list(forgetting.keys())
            values = list(forgetting.values())
            colors = ['red' if v > 0.05 else 'orange' if v > 0.02 else 'green' for v in values]
            ax3.barh(tasks, values, color=colors, alpha=0.7)
            ax3.axvline(x=0, color='black', linestyle='--', linewidth=1)
            ax3.set_xlabel('Forgetting (max - final performance)')
            ax3.set_title('Catastrophic Forgetting per Task')
            ax3.grid(True, alpha=0.3, axis='x')
        
        # 4. Summary metrics
        ax4 = axes[1, 1]
        ax4.axis('off')
        
        bwt = self.compute_backward_transfer()
        avg_forgetting = np.mean(list(forgetting.values())) if forgetting else 0
        max_forgetting = max(forgetting.values()) if forgetting else 0
        
        summary_text = f"""
        Continual Learning Summary
        {'='*40}
        
        Tasks Learned: {len(self.task_order)}
        Task Order: {[self.task_names[i] for i in self.task_order]}
        
        Backward Transfer (BWT): {bwt:+.4f}
        {'  → Positive transfer' if bwt > 0 else '  → Catastrophic forgetting'}
        
        Average Forgetting: {avg_forgetting:.4f}
        Maximum Forgetting: {max_forgetting:.4f}
        
        Final Performance:
        """
        
        if self.task_order:
            final_idx = self.task_order[-1]
            for i, task in enumerate(self.task_names):
                perf = self.performance_matrix[final_idx, i]
                if perf > 0:
                    summary_text += f"  {task}: {perf:.4f}\n"
        
        ax4.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center')
        
        plt.tight_layout()
        plt.savefig('catastrophic_forgetting_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()

# Simulate catastrophic forgetting
def simulate_forgetting_patterns():
    """Simulate different forgetting patterns."""
    tasks = ['Task A', 'Task B', 'Task C', 'Task D']
    tracker = ForgettingTracker(tasks)
    
    # Simulate sequential learning with forgetting
    # Task A: learn to 0.85
    tracker.record_performance(0, {'Task A': 0.85})
    
    # Task B: learn to 0.82, A drops to 0.65 (catastrophic forgetting)
    tracker.record_performance(1, {'Task A': 0.65, 'Task B': 0.82})
    
    # Task C: learn to 0.79, previous tasks continue to drop
    tracker.record_performance(2, {'Task A': 0.55, 'Task B': 0.70, 'Task C': 0.79})
    
    # Task D: learn to 0.80, but with replay buffer, forgetting reduced
    tracker.record_performance(3, {'Task A': 0.60, 'Task B': 0.73, 'Task C': 0.76, 'Task D': 0.80})
    
    tracker.visualize_forgetting()
    
    # Print metrics
    print("\nForgetting Metrics:")
    print(f"Backward Transfer: {tracker.compute_backward_transfer():+.4f}")
    print("\nPer-task Forgetting:")
    for task, forget in tracker.compute_forgetting_measure().items():
        print(f"  {task}: {forget:.4f}")

simulate_forgetting_patterns()

## 7. Complete Continual Learning System

In [ ]:
class ContinualLearningSystem:
    """Complete system combining EWC, replay, and mixing."""
    
    def __init__(
        self,
        model: nn.Module,
        task_names: List[str],
        method: str = 'ewc+replay',  # 'naive', 'ewc', 'replay', 'ewc+replay'
        ewc_lambda: float = 0.4,
        replay_size: int = 5000,
        mixing_ratio: float = 0.2
    ):
        self.model = model
        self.task_names = task_names
        self.method = method
        
        # Initialize components based on method
        self.ewc = None
        self.replay_buffer = None
        self.data_mixer = None
        
        if 'ewc' in method:
            self.ewc = EWC(model, ewc_lambda)
        
        if 'replay' in method:
            self.replay_buffer = ExperienceReplayBuffer(max_size=replay_size)
            self.data_mixer = ContinualLearningDataMixer(
                self.replay_buffer,
                mixing_ratio=mixing_ratio,
                strategy='adaptive'
            )
        
        # Tracking
        self.forgetting_tracker = ForgettingTracker(task_names)
        self.current_task_idx = -1
    
    def train_task(
        self,
        task_idx: int,
        train_loader: any,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 3
    ):
        """
        Train on a new task with continual learning.
        
        Args:
            task_idx: Index of task to learn
            train_loader: Training data for the task
            optimizer: Optimizer
            num_epochs: Number of training epochs
        """
        self.current_task_idx = task_idx
        task_name = self.task_names[task_idx]
        
        print(f"\n{'='*60}")
        print(f"Training Task {task_idx}: {task_name}")
        print(f"Method: {self.method}")
        print(f"{'='*60}")
        
        self.model.train()
        
        for epoch in range(num_epochs):
            epoch_loss = 0
            num_batches = 0
            
            for inputs, targets in train_loader:
                # Mix with replay if available
                if self.data_mixer is not None and epoch > 0:  # Start replay after first epoch
                    inputs, targets, task_ids = self.data_mixer.get_mixed_batch(
                        inputs, targets, task_name
                    )
                
                optimizer.zero_grad()
                
                # Forward pass
                outputs = self.model(inputs)
                task_loss = F.cross_entropy(outputs, targets)
                
                # Add EWC penalty
                if self.ewc is not None and len(self.ewc.task_fisher) > 0:
                    ewc_loss = self.ewc.compute_ewc_loss()
                    total_loss = task_loss + ewc_loss
                else:
                    total_loss = task_loss
                
                # Backward pass
                total_loss.backward()
                optimizer.step()
                
                # Store in replay buffer
                if self.replay_buffer is not None:
                    self.replay_buffer.add_batch(task_name, inputs.detach(), targets.detach())
                
                epoch_loss += total_loss.item()
                num_batches += 1
            
            avg_loss = epoch_loss / num_batches
            print(f"  Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
        
        # Compute Fisher information after training
        if self.ewc is not None:
            print(f"\nComputing Fisher information for {task_name}...")
            self.ewc.compute_fisher_information(task_name, train_loader, num_samples=1000)
    
    def evaluate_all_tasks(self, test_loaders: Dict[str, any]) -> Dict[str, float]:
        """Evaluate model on all learned tasks."""
        self.model.eval()
        results = {}
        
        with torch.no_grad():
            for i, task_name in enumerate(self.task_names):
                if i > self.current_task_idx:  # Haven't learned yet
                    continue
                
                if task_name not in test_loaders:
                    continue
                
                correct = 0
                total = 0
                
                for inputs, targets in test_loaders[task_name]:
                    outputs = self.model(inputs)
                    predictions = outputs.argmax(dim=-1)
                    correct += (predictions == targets).sum().item()
                    total += targets.size(0)
                
                accuracy = correct / total if total > 0 else 0
                results[task_name] = accuracy
        
        # Record in tracker
        self.forgetting_tracker.record_performance(self.current_task_idx, results)
        
        # Update data mixer
        if self.data_mixer is not None:
            for task, perf in results.items():
                self.data_mixer.update_performance(task, perf)
        
        return results
    
    def get_summary(self) -> Dict:
        """Get continual learning summary."""
        summary = {
            'method': self.method,
            'tasks_learned': self.current_task_idx + 1,
            'backward_transfer': self.forgetting_tracker.compute_backward_transfer(),
            'forgetting': self.forgetting_tracker.compute_forgetting_measure()
        }
        
        if self.replay_buffer is not None:
            summary['replay_stats'] = self.replay_buffer.get_statistics()
        
        if self.ewc is not None:
            summary['ewc_stats'] = self.ewc.get_importance_statistics()
        
        return summary

print("Complete continual learning system implemented")

## 8. Best Practices & Recommendations

### Method Selection Guide

| Method | Best For | Pros | Cons |
|--------|----------|------|------|
| **EWC** | Memory-constrained, many tasks | No data storage | Requires task boundaries |
| **Replay** | Few tasks, good memory | Simple, effective | Memory overhead |
| **EWC + Replay** | Best performance | Combines benefits | Most complex |
| **Data Mixing** | LLM fine-tuning | Simple, effective | Needs old data access |
| **LoRA** | Large models | Parameter efficient | May still forget |

### Data Mixing Guidelines

**Recommended Mixing Ratios:**
- **80/20 (new/old)**: Standard for similar domains
- **90/10**: When new task is very different
- **70/30**: When forgetting is severe
- **50/50**: For equal importance of tasks

**LLM Fine-Tuning Best Practices:**
1. **Always mix base pre-training data** (at least 10%)
2. **Include general instruction data** if doing task-specific tuning
3. **Monitor base capabilities** (reasoning, knowledge, language)
4. **Use temperature-scaled sampling** for imbalanced datasets

### EWC Hyperparameters

**Lambda (regularization strength):**
- Too low (< 0.01): Insufficient protection, forgetting occurs
- Optimal (0.1 - 1.0): Balance between plasticity and stability
- Too high (> 10): Model can't learn new tasks

**Fisher samples:**
- Minimum: 500 examples
- Recommended: 1000-2000 examples
- More samples = better importance estimation

### Replay Buffer Design

**Buffer Size:**
- Rule of thumb: 1-5% of total training data
- Minimum per task: 100-500 examples
- More diverse tasks need larger buffers

**Sampling Strategy:**
- **Reservoir sampling**: Online, uniform probability
- **Class-balanced**: For imbalanced classification
- **Hard examples**: Store challenging cases
- **Diverse selection**: Maximize coverage

### Prevention Strategies

1. **Architecture choices:**
   - Parameter-efficient methods (LoRA, adapters)
   - Task-specific heads with frozen backbone
   - Progressive neural networks

2. **Training strategies:**
   - Gradual unfreezing (start with last layers)
   - Lower learning rate for old knowledge
   - Knowledge distillation from previous version

3. **Monitoring:**
   - Regular evaluation on all tasks
   - Track backward transfer metric
   - Alert on >5% performance drop

### Common Pitfalls

1. **Not monitoring forgetting:** Evaluate old tasks regularly
2. **Insufficient replay data:** Use at least 1% of training size
3. **Wrong mixing ratio:** Adapt based on forgetting
4. **Ignoring task similarity:** Related tasks need less protection
5. **No baseline:** Always compare to single-task and naive sequential

## 9. Benchmarks & Real-World Results

### Academic Benchmarks

**Split MNIST/CIFAR (Sequential Tasks):**
- Naive fine-tuning: 40-60% on old tasks (severe forgetting)
- EWC: 75-85% retention
- Replay (5% buffer): 85-90% retention
- EWC + Replay: 90-95% retention

**Permuted MNIST:**
- 10 tasks, input pixels permuted
- EWC: 82% average accuracy
- Progressive nets: 95% (but 10x parameters)

**CORe50 (Continual Object Recognition):**
- 50 object classes, 11 sessions
- Experience Replay: 65-70% final accuracy
- iCaRL (class-incremental): 72%

### LLM Continual Learning

**Domain Adaptation:**
- Without mixing: 30-50% degradation on general tasks
- With 20% base data: <10% degradation
- With 40% base data: <5% degradation

**Instruction Tuning:**
- FLAN-T5: Trained on 1800+ tasks, minimal forgetting
- Method: Proportional mixing with temperature = 0.7
- Result: Strong multi-task performance

**Continual Pre-training:**
- Adding new domain (e.g., code, medical)
- 80/20 mixing: Maintains general capabilities
- Pure domain training: Loses reasoning abilities

### Industry Applications

**Production ML Systems:**
- Recommendation systems: Continual learning with weekly updates
- Fraud detection: Adapt to new fraud patterns without forgetting old
- Content moderation: Add new policy rules without retraining from scratch

**Model Updates:**
- Typical approach: 70-90% new data, 10-30% historical data
- Evaluation: Monitor key metrics on old test sets
- Rollback: If >10% degradation on any critical metric

### Performance Patterns

**Forgetting vs. Task Similarity:**
- Similar tasks (sentiment → emotion): 5-15% forgetting
- Different domains (text → code): 30-50% forgetting
- Opposite tasks (positive → negative): 60-80% forgetting

**Replay Buffer Size Impact:**
- 0% (no replay): 50-70% forgetting
- 1% buffer: 20-30% forgetting
- 5% buffer: 10-15% forgetting
- 10% buffer: 5-10% forgetting
- Diminishing returns beyond 10%

## Summary

Catastrophic forgetting is a fundamental challenge in continual learning:

**Key Concepts:**
1. Neural networks forget old knowledge when learning new tasks
2. EWC protects important parameters using Fisher Information
3. Experience replay stores representative old examples
4. Data mixing (80/20 rule) balances new and old learning
5. Multiple strategies can be combined for best results

**Method Selection:**
- **Limited memory?** Use EWC
- **Few tasks?** Use replay
- **Best performance?** Combine EWC + replay
- **LLM fine-tuning?** Use data mixing

**Critical Practices:**
- Always monitor old task performance
- Use at least 10-20% replay/old data
- Track backward transfer metric
- Consider parameter-efficient methods (LoRA)

**Next Steps:**
- Implement method combinations
- Experiment with mixing ratios
- Try parameter-efficient fine-tuning (Notebooks 29-30)
- Explore multi-task learning (Notebook 45)